# 07 - Clustering & Spatial Domains

## Learning objectives
1. Build a neighbor graph, run UMAP, and cluster spots with Leiden.
2. Visualize clusters in UMAP space and over the histology.
3. Distinguish **transcriptomic clusters** from **morphological regions**.
4. Find per-cluster marker genes and export a table.

## Concept
Clustering groups spots with similar expression - **unsupervised segmentation in feature
space**. UMAP is a 2D map for visualization (like t-SNE). Leiden finds communities in the
nearest-neighbor graph. Mapping clusters back onto tissue reveals spatial domains.


In [ ]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd

adata = st.load_adata('adata_qc.h5ad')
assert 'X_pca' in adata.obsm, 'Run notebook 04 first (PCA is computed there).'
adata


### Neighbor graph + UMAP + Leiden
We use the PCA from notebook 04 (first ~30 PCs), build a kNN graph, embed with UMAP, and
cluster with Leiden. All seeded for reproducibility.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, random_state=st.SEED)
sc.tl.umap(adata, random_state=st.SEED)
st.run_leiden(adata, resolution=1.0, key_added='clusters', seed=st.SEED)
print('Number of Leiden clusters:', adata.obs['clusters'].nunique())
adata.obs['clusters'].value_counts().sort_index()


### Clusters in UMAP space

In [ ]:
sc.pl.umap(adata, color='clusters', legend_loc='on data', title='Leiden clusters (UMAP)')


### Clusters over the histology (spatial domains)

In [ ]:
sq.pl.spatial_scatter(adata, color='clusters', size=1.4)


**Expected output:** ~10-20 clusters that, on the tissue, form **spatially coherent
domains** mirroring brain anatomy (cortical layers, white matter, hippocampus, etc.) -
even though clustering used only expression, not position.

### Transcriptomic clusters vs morphological regions
These clusters come purely from expression. They often align with morphology you can see
in the H&E, but **not always**: two regions can look alike on H&E yet differ molecularly
(and vice versa). That mismatch is scientifically interesting - and the motivation for
notebook 10, where we ask how well morphology predicts the transcriptomic cluster.

### Spatial graph (optional, for spatial methods)
Squidpy can build a graph on the **hex grid** (spatial neighbors), used by spatially aware
methods like Moran's I in notebook 08.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type='grid', n_neighs=6)
print('spatial graph stored in adata.obsp:', [k for k in adata.obsp.keys()])


### Marker genes per cluster
`rank_genes_groups` runs a one-vs-rest test per cluster (Wilcoxon) to find genes that
define it - the molecular 'label' for each segment. We export the top markers to CSV.

In [ ]:
sc.tl.rank_genes_groups(adata, 'clusters', method='wilcoxon')
markers = sc.get.rank_genes_groups_df(adata, group=None)
top_markers = (markers.sort_values(['group', 'scores'], ascending=[True, False])
                      .groupby('group').head(10).reset_index(drop=True))
markers_csv = st.outputs_dir() / 'cluster_markers.csv'
top_markers.to_csv(markers_csv, index=False)
print('Wrote', markers_csv)
top_markers.head(20)


In [ ]:
# Quick look: top-3 markers per cluster on a dotplot.
top3 = (top_markers.groupby('group').head(3)['names'].tolist())
top3 = st.genes_present(adata, dict.fromkeys(top3))  # dedupe + existence check
sc.pl.dotplot(adata, var_names=top3[:25], groupby='clusters')


In [ ]:
saved = st.save_adata(adata, 'adata_clustered.h5ad')
print('Wrote', saved)


## Common pitfalls
- No `leidenalg`/`igraph` installed -> Leiden fails (the helper falls back; install both).
- Changing `resolution` changes cluster count - there is no single 'true' number.
- Reading cluster numbers as cell types - they are spot groups; annotate via markers (nb 11).

## Interpretation
Unsupervised clustering recovered tissue domains from expression alone, and each domain has
a marker-gene signature you can export and inspect.

## What this means biologically
The brain is spatially organized into regions with distinct molecular programs. That the
clusters reproduce known anatomy validates the pipeline and lets us name domains by their
marker genes.

---
**Next:** `08_spatially_variable_genes.ipynb`.
